# 06 - Digital image fundamentals

**Syllabus:** image perception (light, luminance, brightness, contrast) · fields that use digital
image processing · a simple image formation model · image sampling and quantization ·
basic relationships between pixels · types of images

This is the unit everything else rests on. Four of the five tasks here are things you will
re-use in later notebooks: quantisation (compression), subsampling (pyramids), neighbourhoods
(every filter and every morphological operator), and connected components (segmentation).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
# the graders are the test harness -- everything else is written out in full
from cvlab import check_components, check_distance, check_downsample, check_neighbours, check_quantize


## 1. Perception: luminance, brightness, contrast

Three words that get used interchangeably and must not be:

| term | what it is | units |
|---|---|---|
| **luminance** | physical light energy leaving a surface toward the eye | cd/m² — measurable |
| **brightness** | the *perceived* intensity of that light | subjective, no unit |
| **contrast** | difference in luminance between a region and its surround | ratio, dimensionless |

The gap between luminance and brightness is why image processing is not just arithmetic.
The human visual system reports **ratios**, not differences — **Weber's law**: the smallest
detectable change ΔI is proportional to the background I. Doubling 10 to 20 is as visible as
doubling 100 to 200, even though the second change is 10× bigger in absolute terms.

Two consequences you can see directly:

In [ ]:
# Simultaneous contrast: the four inner squares are the SAME pixel value.
panel = np.zeros((160, 640), np.uint8)
for i, bg in enumerate([0, 80, 160, 240]):
    panel[:, i*160:(i+1)*160] = bg
    panel[50:110, i*160+50:i*160+110] = 128
_imgs, _titles = [panel], ["all four inner squares are exactly 128"]
fig, axes = plt.subplots(1, 1, figsize=(11 * 1, 11), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
print("proof:", [int(panel[80, i*160+80]) for i in range(4)])

In [ ]:
# Mach bands: uniform strips, but you will see bright/dark lines at each boundary
# that are NOT in the data. Your retina's lateral inhibition invents them.
bands = np.repeat(np.arange(8) * 32, 80).astype(np.uint8)[None, :].repeat(200, 0)
_imgs, _titles = [bands], ["8 uniform strips - look at the edges"]
fig, axes = plt.subplots(1, 1, figsize=(11 * 1, 11), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(11, 2)); plt.plot(bands[100]); plt.title("the actual pixel values: flat steps")
plt.tight_layout(); plt.show()

**Why this matters practically:** it is why gamma (notebook 04) exists, why we quantise
*perceptually* rather than uniformly, why JPEG throws away chroma before luma, and why
"the histogram looks fine" is not the same as "the image looks fine".

### Fields that use digital image processing

| field | what it looks at | typical operation |
|---|---|---|
| medicine | X-ray, CT, MRI, histology | enhancement, segmentation, registration |
| remote sensing | satellite/multispectral | classification, pseudo-colour, band maths |
| industrial inspection | PCB, weld, fabric | defect detection, morphology, template matching |
| astronomy | long-exposure telescope | noise reduction, deconvolution, stacking |
| forensics / security | fingerprints, iris, faces, CCTV | biometrics, restoration, tracking |
| document processing | scans, forms, handwriting | binarisation, deskew, OCR |
| consumer photography | phone cameras | denoise, HDR, super-resolution, portrait blur |
| autonomous vehicles | camera + lidar | detection, depth, motion |

Your assignments 1-4 are the enhancement primitives all eight of those columns depend on.

---
## 2. A simple image formation model

$$f(x, y) = i(x, y) \cdot r(x, y)$$

The image is **illumination** × **reflectance**. Illumination `i` varies slowly and smoothly
(a lamp, the sun). Reflectance `r` is the object property you usually want, and it changes
fast at object boundaries. `0 < i < ∞` and `0 < r < 1`.

The multiplication is exactly why uneven lighting is hard to remove with subtraction, and why
taking the **log** turns it into an additive problem that a high-pass filter can separate —
that is homomorphic filtering, which you will build in notebook 10.

In [ ]:
import cv2
# Build an image from the model, then see why a global threshold fails on it.
r = cv2.imread("images/sudoku.png", cv2.IMREAD_GRAYSCALE).astype(np.float64) / 255      # reflectance
yy, xx = np.mgrid[0:r.shape[0], 0:r.shape[1]]
i = 0.25 + 1.35 * np.exp(-((xx - 80)**2 + (yy - 60)**2) / (2 * 260**2))   # a lamp, top-left
f = np.clip(i * r, 0, 1)
_imgs, _titles = [(r*255).astype(np.uint8), (i/i.max()*255).astype(np.uint8), (f*255).astype(np.uint8)], ["reflectance r(x,y)", "illumination i(x,y)", "observed f = i*r"]
fig, axes = plt.subplots(1, 3, figsize=(3.6 * 3, 3.6), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

obs = (f*255).astype(np.uint8)
_, glob = cv2.threshold(obs, obs.mean(), 255, cv2.THRESH_BINARY)
_imgs, _titles = [obs, glob], ["observed", "one global threshold - fails on the dark corner"]
fig, axes = plt.subplots(1, 2, figsize=(4.5 * 2, 4.5), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

---
## 3. Sampling and quantisation

Digitising a continuous image is two independent discretisations:

- **sampling** = discretising the *coordinates* (how many pixels). Too few → **aliasing**.
- **quantisation** = discretising the *amplitude* (how many grey levels). Too few → **false contouring**.

### Task 1 - quantisation

Uniform requantisation to `levels` grey levels, spanning the full 0..255 range.
Spec: with `step = 256 // levels`, the bin index is `gray // step`, and bin `k` maps to
`round(k * 255 / (levels - 1))`. So `levels=2` gives `{0, 255}`, `levels=4` gives
`{0, 85, 170, 255}`, and `levels=256` is the identity.

Assume `levels` divides 256 and is at least 2. **Watch the dtype** — `bin_index * 255`
overflows `uint8`, and this is the single most common bug in this notebook.

In [ ]:
def my_quantize(gray, levels):
    """Uniformly requantise to `levels` grey levels, spanning 0..255.

    Parameters
    ----------
    gray   : (H, W) uint8 array
    levels : int >= 2 that divides 256 (2, 4, 8, 16, 32, 64, 128, 256)

    Returns
    -------
    (H, W) uint8 array with at most `levels` distinct values, the smallest 0 and the largest 255.
    levels=256 returns the input unchanged.
    """
    # TODO: bin index, then map the index back onto the full range
    raise NotImplementedError

In [ ]:
check_quantize(my_quantize)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# No cv2.quantize exists -- it is a LUT again.
import cv2, numpy as np
levels = 4
step = 256 // levels
lut = np.round((np.arange(256) // step) * 255 / (levels - 1)).astype(np.uint8)
g = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
assert np.array_equal(cv2.LUT(g, lut), my_quantize(g, levels)), "disagrees with the LUT route"
print("match. Note how the LUT makes the uint8 overflow trap disappear:")
print("np.arange(256) is int64, so the arithmetic is safe before the final cast.")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

`step = 256 // levels`, then `idx = gray // step`.

Now the trap: `idx` is still `uint8`, so `idx * 255` wraps around. Cast it wider first —
`.astype(np.int64)` or divide by 255.0 — before scaling by `255 / (levels - 1)`.
Then `.round().astype(np.uint8)`.

</details>

In [ ]:
import cv2
_imgs, _titles = list([my_quantize(cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE), L) for L in [256, 32, 8, 4, 2]]), [f"{L} levels ({int(np.log2(L))} bits)" for L in [256, 32, 8, 4, 2]]
_cols = 5; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.0 * _cols, 3.0 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

# False contouring is easiest to see on a smooth gradient, not on a detailed photo.
grad = np.tile(np.linspace(0, 255, 512).astype(np.uint8), (160, 1))
_imgs, _titles = list([my_quantize(grad, L) for L in [256, 16, 8, 4]]), [f"{L} levels" for L in [256, 16, 8, 4]]
fig, axes = plt.subplots(1, len(_imgs), figsize=(3.2 * len(_imgs), 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

Detail hides quantisation; smooth gradients expose it. That asymmetry is the reason
**dithering** works — deliberately adding noise before quantising trades false contours for
grain, and the eye prefers grain.

### Task 2 - sampling

Naive subsampling: keep every `factor`-th row and column, no pre-filtering. The point of doing
it the naive way is to *see* aliasing.

In [ ]:
def my_resize_nearest(gray, factor):
    """Subsample by keeping every `factor`-th row and column. No pre-filtering.

    Parameters
    ----------
    gray   : (H, W) uint8 array
    factor : int >= 1

    Returns
    -------
    uint8 array of shape (ceil(H/factor), ceil(W/factor)) holding the pixels at
    row/column indices 0, factor, 2*factor, ... Values are copied, never averaged.
    """
    # TODO: one slice does this
    raise NotImplementedError

In [ ]:
check_downsample(my_resize_nearest)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.resize(src, dsize, fx=, fy=, interpolation=)
#   dsize is (WIDTH, HEIGHT) -- reversed from numpy's (rows, cols). Classic exam trap.
import cv2, numpy as np
g = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
near = cv2.resize(g, (g.shape[1]//4, g.shape[0]//4), interpolation=cv2.INTER_NEAREST)
area = cv2.resize(g, (g.shape[1]//4, g.shape[0]//4), interpolation=cv2.INTER_AREA)
print("your naive slice :", my_resize_nearest(g, 4).shape)
print("cv2 INTER_NEAREST:", near.shape, " (same idea, still aliases)")
print("cv2 INTER_AREA   :", area.shape, " (averages first -- the correct way to shrink)")
print("\ninterpolation flags: INTER_NEAREST, INTER_LINEAR (default), INTER_CUBIC,")
print("                     INTER_AREA (shrinking), INTER_LANCZOS4")
print("cv2.pyrDown(src) / cv2.pyrUp(src) do a blur-then-halve / double in one call.")
_imgs, _titles = [my_resize_nearest(g, 4), near, area], ["yours", "INTER_NEAREST", "INTER_AREA"]
fig, axes = plt.subplots(1, 3, figsize=(3.2 * 3, 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Strided slicing: `gray[::factor, ::factor]`. That is the whole function.

If the shape check fails, remember Python slicing already rounds up for you — a 6-row
image with factor 3 gives rows 0 and 3, so 2 rows.

</details>

In [ ]:
# Aliasing: a fine radial pattern subsampled with and without a pre-blur.
yy, xx = np.mgrid[-256:256, -256:256].astype(np.float64)
zone = (127 * (1 + np.sin((xx**2 + yy**2) / 900))).astype(np.uint8)
naive   = my_resize_nearest(zone, 6)
blurred = my_resize_nearest(cv2.GaussianBlur(zone, (0, 0), 2.5), 6)
_imgs, _titles = [zone, naive, blurred], ["original pattern", "naive 1:6 - false rings (ALIASING)", "blur first, then 1:6"]
fig, axes = plt.subplots(1, 3, figsize=(4 * 3, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

Those extra rings in the middle image are not in the original. They are frequencies above the
**Nyquist limit** folding back down into the visible range. The fix is always the same: low-pass
filter *before* you subsample. `cv2.resize` and `cv2.pyrDown` do this for you; a raw slice does not.

This is the same theorem that says you must sample at more than twice the highest frequency
present. It comes back formally in notebook 10.

In [ ]:
import cv2
# Sampling and quantisation are independent axes. Vary both.
g = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for row, f in enumerate([1, 4, 16]):
    for col, L in enumerate([256, 16, 4, 2]):
        axes[row, col].imshow(my_quantize(my_resize_nearest(g, f), L), cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(f"1:{f} sampling, {L} levels", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout(); plt.show()

---
## 4. Basic relationships between pixels

The vocabulary the rest of the course assumes. For a pixel `p = (r, c)`:

- **N₄(p)** — the 4 horizontal/vertical neighbours
- **N_D(p)** — the 4 diagonal neighbours
- **N₈(p)** — all 8, i.e. N₄ ∪ N_D

Built on top of those:

- **adjacency** — two pixels are adjacent if they are neighbours *and* their values both lie in
  a set `V` of "similar" values (for a binary image, `V = {255}`).
  **4-adjacency**, **8-adjacency**, and **m-adjacency** (mixed: 4-adjacent, *or* diagonal with no
  4-connected shortcut) — m-adjacency exists purely to kill the ambiguous multiple paths that
  8-adjacency creates.
- **path** — a sequence of adjacent pixels. **connected** — a path exists.
  **connected component** — a maximal set of mutually connected pixels.
- **region** — a connected set. **boundary** — pixels of a region with at least one
  background neighbour.

### Distance measures

| name | formula | unit ball shape |
|---|---|---|
| **D₄** city-block | \|Δr\| + \|Δc\| | diamond |
| **D₈** chessboard | max(\|Δr\|, \|Δc\|) | square |
| **euclidean** | √(Δr² + Δc²) | circle |

Always `D₈ ≤ euclidean ≤ D₄`.

In [ ]:
def my_neighbours(shape, p, kind):
    """In-bounds neighbour coordinates of pixel p in an image of the given shape.

    Parameters
    ----------
    shape : (H, W) tuple
    p     : (row, col) tuple
    kind  : "N4" (4 horizontal/vertical), "ND" (4 diagonal), or "N8" (all 8)

    Returns
    -------
    list of (row, col) tuples, excluding p itself and anything outside the image.
    Order does not matter -- the grader compares sets.
    """
    # TODO: build the offset list for `kind`, add it to p, drop out-of-bounds results
    raise NotImplementedError

In [ ]:
check_neighbours(my_neighbours)

In [ ]:
# ---- The OpenCV connection ---------------------------------------------------
# There is no cv2.neighbours -- but N4 and N8 ARE the two structuring elements
# every morphological operation is built on. cv2 spells them like this:
import cv2
print("MORPH_CROSS 3x3  = N4 + centre:\n", cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3)))
print("\nMORPH_RECT 3x3  = N8 + centre:\n", cv2.getStructuringElement(cv2.MORPH_RECT, (3,3)))
print("\nMORPH_ELLIPSE 5x5:\n", cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5)))
print("\ncv2.getStructuringElement(shape, ksize, anchor=None) -- you will use this in notebook 12.")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Keep three offset lists and pick one by `kind`:
`d4 = [(-1,0),(1,0),(0,-1),(0,1)]`, `dd = [(-1,-1),(-1,1),(1,-1),(1,1)]`, and `N8 = d4 + dd`.

Then one list comprehension adding each offset to `p`, with a bounds condition
`0 <= r < shape[0] and 0 <= c < shape[1]`. Do not use `if r > 0` style tests — you need
*both* ends of *both* axes, which is why the corner cases are in the grader.

</details>

In [ ]:
def my_distance(p, q, metric):
    """Distance between two pixel coordinates.

    Parameters
    ----------
    p, q   : (row, col) tuples (may hold negative values)
    metric : "D4" (city-block), "D8" (chessboard), or "euclidean"

    Returns
    -------
    float or int. Zero when p == q, symmetric in p and q, never negative.
    """
    # TODO: take absolute coordinate differences, then apply the right formula
    raise NotImplementedError

In [ ]:
check_distance(my_distance)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.distanceTransform(src, distanceType, maskSize) gives, for every non-zero pixel,
# the distance to the NEAREST ZERO pixel -- your metric applied over a whole image.
#   cv2.DIST_L1 = D4 (city-block)   cv2.DIST_C = D8 (chessboard)   cv2.DIST_L2 = euclidean
import cv2, numpy as np
b = np.zeros((41, 41), np.uint8); b[5:36, 5:36] = 255
for name, flag in [("DIST_L1 (D4)", cv2.DIST_L1), ("DIST_C (D8)", cv2.DIST_C),
                   ("DIST_L2 (euclid)", cv2.DIST_L2)]:
    d = cv2.distanceTransform(b, flag, 5)
    print(f"{name:18} max distance inside the square = {d.max():.2f}")
print("\nAnd your own function agrees on a single pair:")
print("  D4", my_distance((0,0),(3,4),"D4"), " D8", my_distance((0,0),(3,4),"D8"),
      " euclid", round(my_distance((0,0),(3,4),"euclidean"), 3))

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Compute `dr = abs(p[0]-q[0])` and `dc = abs(p[1]-q[1])` **first** — taking the absolute
value up front is what makes all three formulas one-liners and makes symmetry automatic.

D4 is the sum, D8 is the max, euclidean is `np.hypot(dr, dc)`.

</details>

In [ ]:
# The three unit balls, drawn from your own function.
for metric in ["D4", "D8", "euclidean"]:
    d = np.array([[my_distance((15, 15), (r, c), metric) for c in range(31)] for r in range(31)])
    plt.figure(figsize=(3.1, 3.1))
    plt.imshow(d, cmap="viridis"); plt.contour(d, levels=[5, 10], colors="w")
    plt.title(f"{metric}: contours at 5 and 10"); plt.axis("off"); plt.tight_layout(); plt.show()

### Task 5 - connected components

Count the connected white regions of a binary image. This is your first real *algorithm*:
a flood fill (BFS or DFS) over the neighbour relation you just wrote.

The classic test is a diagonal pair of white pixels: **two** components under 4-connectivity,
**one** under 8-connectivity. Your function must reproduce that, which means the connectivity
argument has to actually reach the neighbour lookup.

In [ ]:
def my_connectedComponents(binary, connectivity):
    """Number of connected components of white (non-zero) pixels.

    Parameters
    ----------
    binary       : (H, W) uint8 array, 0 = background, 255 = foreground
    connectivity : 4 or 8

    Returns
    -------
    int. 0 for an all-black image. Holes inside a region are not counted separately.
    """
    # TODO: keep a `seen` mask; for each unvisited white pixel, flood fill and add 1
    raise NotImplementedError

In [ ]:
check_components(my_connectedComponents)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.connectedComponents(image, connectivity=8) -> (num_labels, label_image)
#   num_labels INCLUDES the background as label 0, so subtract 1 for object count.
# The richer version gives you stats for free:
# cv2.connectedComponentsWithStats(...) -> (n, labels, stats, centroids)
#   stats[i] = [x, y, w, h, area]  -- bounding box and pixel count per component
import cv2, numpy as np
b = ((cv2.imread("images/sudoku.png", cv2.IMREAD_GRAYSCALE) < 100) * 255).astype(np.uint8)
b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
n, labels, stats, cents = cv2.connectedComponentsWithStats(b, connectivity=8)
print("cv2 object count:", n - 1, "  yours:", my_connectedComponents(b, 8))
assert n - 1 == my_connectedComponents(b, 8), "disagrees with cv2"
big = stats[1:][stats[1:, 4].argsort()[::-1]][:5]
print("\n5 largest components  [x, y, w, h, area]:")
for row in big: print("   ", row.tolist())

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Structure:

1. `seen = np.zeros(binary.shape, bool)` and `n = 0`
2. loop over every pixel; skip it if it is black or already seen
3. otherwise `n += 1` and flood fill from there, marking everything reachable as seen
4. return `n`

For the flood fill use `collections.deque` as a queue (BFS) and call your own
`my_neighbours(binary.shape, p, "N4" if connectivity == 4 else "N8")`. Mark a pixel `seen`
when you *enqueue* it, not when you dequeue it, or it lands in the queue many times.

</details>

In [ ]:
import cv2
# Count the digits in the sudoku grid, then see where your count comes from.
b = ((cv2.imread("images/sudoku.png", cv2.IMREAD_GRAYSCALE) < 100) * 255).astype(np.uint8)
b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
print("4-connected components:", my_connectedComponents(b, 4))
print("8-connected components:", my_connectedComponents(b, 8))
n, lab = cv2.connectedComponents(b, connectivity=8)
_imgs, _titles = [b, (lab * (255 // max(n,1)) % 255).astype(np.uint8)], [f"binary", f"cv2 labels ({n-1} components)"]
fig, axes = plt.subplots(1, 2, figsize=(4.5 * 2, 4.5), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

---
## Think about it

1. In the simultaneous-contrast panel, all four squares are 128. Explain why the leftmost looks
   brightest using Weber's law. Then: what does this imply about trusting your eye when you
   judge whether an enhancement "worked"?
2. Sampling and quantisation both throw away information. For a fixed **storage budget** —
   say 64 kB — is it better to keep more pixels at fewer levels, or fewer pixels at more levels?
   Answer for a text scan and for a smooth portrait, and explain the difference.
3. Under 8-connectivity, find a small binary pattern where two white pixels are connected by
   two different paths and the "region boundary" becomes ambiguous. This is exactly the case
   m-adjacency was invented to remove. Draw it as a numpy array.
4. Prove `D₈(p,q) ≤ euclidean(p,q) ≤ D₄(p,q)` from the formulas. One line each.
5. `my_connectedComponents` on a 512×512 image with a pure-Python flood fill takes noticeable time.
   Where does it go? What does `cv2.connectedComponents` do differently? (Search: two-pass
   labelling, union-find.)

In [ ]:
# Scratch space.

---
## Stretch

1. **Dithering.** Add uniform noise of ±half-a-step before quantising to 4 levels, and compare
   against plain quantisation on the smooth gradient. Then implement Floyd-Steinberg error
   diffusion and compare all three.
2. **A proper pyramid.** Write `pyramid(gray, n)` returning `n` levels, each a correctly
   pre-filtered half-size version. Compare against `cv2.pyrDown`. Show that naive `my_resize_nearest`
   diverges from it more at every level.
3. **Distance transform.** For a binary image, compute for every background pixel its D₄ distance
   to the nearest foreground pixel — by BFS from all foreground pixels at once. Compare against
   `cv2.distanceTransform`. You have just built the foundation of the watershed algorithm.
4. **m-adjacency.** Implement `m_adjacent(img, p, q, V)` and use it to count components under
   m-connectivity. Show a pattern where m-connectivity and 8-connectivity disagree.

In [ ]:
# Stretch scratch space.

---
## OpenCV API card - resizing, structuring elements, components

```python
dst = cv2.resize(src, dsize, fx=0, fy=0, interpolation=INTER_LINEAR)
```
`dsize` is **(width, height)** — reversed from `src.shape`, which is `(rows, cols)`.
Pass `dsize=None` with `fx`/`fy` to scale by factor instead.
`INTER_NEAREST` (blocky, aliases), `INTER_LINEAR` (default), `INTER_CUBIC`,
`INTER_AREA` (**the correct choice when shrinking**), `INTER_LANCZOS4`.

```python
dst = cv2.pyrDown(src)   # blur then halve -- the anti-aliased downsample
dst = cv2.pyrUp(src)     # double then blur
```

```python
se = cv2.getStructuringElement(shape, ksize, anchor=(-1,-1))
```
`MORPH_RECT` (= N₈ plus centre), `MORPH_CROSS` (= N₄ plus centre), `MORPH_ELLIPSE`.

```python
n, labels = cv2.connectedComponents(image, connectivity=8)
n, labels, stats, centroids = cv2.connectedComponentsWithStats(image, connectivity=8)
```
`n` counts the background as label 0, so objects = `n - 1`.
`stats[i]` is `[x, y, w, h, area]`; `centroids[i]` is `(x, y)`.

```python
dst = cv2.distanceTransform(src, distanceType, maskSize)
```
`DIST_L1` = D₄, `DIST_C` = D₈, `DIST_L2` = euclidean. `maskSize` is 3, 5, or `DIST_MASK_PRECISE`.